In [1]:
import pandas as pd
from pipeline import create_X_y_df

In [2]:
df_full = pd.read_parquet("../../data/churn-prediction-25-26/train.parquet")

In [3]:
df = df_full.copy()

In [52]:
# Second Train Test set
cut_off_date_train = pd.Timestamp("2018-11-05")
cut_off_date_test = pd.Timestamp("2018-11-15")

df_X, df_y = create_X_y_df(df = df,
                                t0=cut_off_date_train,
                                t1=cut_off_date_test)

In [57]:
from skrub import GapEncoder

In [58]:
gec = GapEncoder(n_components=3)

In [59]:
gec.fit(pd.Series(df_X["location"].unique(), name = "location"))

,n_components,3
,batch_size,1024
,gamma_shape_prior,1.1
,gamma_scale_prior,1.0
,rho,0.95
,rescale_rho,False
,hashing,False
,hashing_n_features,4096
,init,'k-means++'
,max_iter,5
,ngram_range,"(2, ...)"


In [60]:
encoded_df = gec.transform(df_X["location"])

In [61]:
encoded_df

,"location: richmond, richland, hammonton","location: sweetwater, philadelphia, snyder","location: opelousas, stroudsburg, pittsfield"
1,1.459847,0.009275,0.828129
448,1.459847,0.009275,0.828129
945,1.459847,0.009275,0.828129
1509,1.459847,0.009275,0.828129
2066,1.459847,0.009275,0.828129
...,...,...,...
24819307,1.603532,0.013054,0.015866
24819309,1.603532,0.013054,0.015866
24819407,1.603532,0.013054,0.015866
24819410,1.603532,0.013054,0.015866


In [62]:
df_a = pd.concat((df_X, encoded_df), axis = 1)

In [63]:
df_a

,status,gender,firstName,level,lastName,userId,ts,auth,page,sessionId,...,method,length,song,artist,time,registration,date,"location: richmond, richland, hammonton","location: sweetwater, philadelphia, snyder","location: opelousas, stroudsburg, pittsfield"
1,200,F,Vianney,paid,Miller,1563081,1538352002000,Logged In,NextSong,20836,...,PUT,238.39302,MiÃÂ©ntele,Los Bunkers,2018-10-01 00:00:02,2018-09-21 03:25:18,2018-10-01 00:00:02,1.459847,0.009275,0.828129
448,200,F,Vianney,paid,Miller,1563081,1538352240000,Logged In,NextSong,20836,...,PUT,263.88853,Animal,Miike Snow,2018-10-01 00:04:00,2018-09-21 03:25:18,2018-10-01 00:04:00,1.459847,0.009275,0.828129
945,200,F,Vianney,paid,Miller,1563081,1538352503000,Logged In,NextSong,20836,...,PUT,273.47546,Mi Corazon Continuara,Lorena,2018-10-01 00:08:23,2018-09-21 03:25:18,2018-10-01 00:08:23,1.459847,0.009275,0.828129
1509,200,F,Vianney,paid,Miller,1563081,1538352776000,Logged In,NextSong,20836,...,PUT,284.86485,The Real Slim Shady,Eminem,2018-10-01 00:12:56,2018-09-21 03:25:18,2018-10-01 00:12:56,1.459847,0.009275,0.828129
2066,200,F,Vianney,paid,Miller,1563081,1538353060000,Logged In,NextSong,20836,...,PUT,304.45669,The People In My Peephole,Claw Hammer,2018-10-01 00:17:40,2018-09-21 03:25:18,2018-10-01 00:17:40,1.459847,0.009275,0.828129
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24819307,200,F,Valeria,paid,Wright,1880093,1541364613000,Logged In,NextSong,2612,...,PUT,243.90485,Let's Make Rock,Dream Evil,2018-11-04 20:50:13,2018-09-01 20:18:00,2018-11-04 20:50:13,1.603532,0.013054,0.015866
24819309,307,F,Valeria,paid,Wright,1880093,1541364614000,Logged In,Thumbs Down,2612,...,PUT,NaN,None,None,2018-11-04 20:50:14,2018-09-01 20:18:00,2018-11-04 20:50:14,1.603532,0.013054,0.015866
24819407,200,F,Valeria,paid,Wright,1880093,1541364856000,Logged In,NextSong,2612,...,PUT,217.57342,Eleanor (Digital Album Version),JET,2018-11-04 20:54:16,2018-09-01 20:18:00,2018-11-04 20:54:16,1.603532,0.013054,0.015866
24819410,307,F,Valeria,paid,Wright,1880093,1541364857000,Logged In,Thumbs Down,2612,...,PUT,NaN,None,None,2018-11-04 20:54:17,2018-09-01 20:18:00,2018-11-04 20:54:17,1.603532,0.013054,0.015866


In [67]:
X_columns = []

for elem in df_a.columns:
    if "location:" in elem:
        X_columns.append(elem)

In [68]:
df_a.columns

Index(['status', 'gender', 'firstName', 'level', 'lastName', 'userId', 'ts',
       'auth', 'page', 'sessionId', 'location', 'itemInSession', 'userAgent',
       'method', 'length', 'song', 'artist', 'time', 'registration', 'date',
       'location: richmond, richland, hammonton',
       'location: sweetwater, philadelphia, snyder',
       'location: opelousas, stroudsburg, pittsfield'],
      dtype='object')

In [69]:
df_b = df_a[X_columns + ["userId"]]


X_columns

['location: richmond, richland, hammonton',
 'location: sweetwater, philadelphia, snyder',
 'location: opelousas, stroudsburg, pittsfield']

In [70]:
df_c = df_b.drop_duplicates()

In [71]:
df_c = df_c.merge(df_y, on = "userId")

In [72]:
X = df_c[X_columns]

y = df_c["Cancellation Confirmation"]

In [90]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

In [74]:
X_train, X_test, y_train, y_test = train_test_split(X, y)

In [93]:
sw = compute_sample_weight("balanced", y_train)

In [94]:
clf = HistGradientBoostingClassifier(
    max_iter=500,
    learning_rate=0.05,
    max_leaf_nodes=63,
    min_samples_leaf=20,
    l2_regularization=0.0,
    random_state=0
)

In [95]:
clf.fit(X_train, y_train, sample_weight=sw)

,loss,'log_loss'
,learning_rate,0.05
,max_iter,500
,max_leaf_nodes,63
,max_depth,None
,min_samples_leaf,20
,l2_regularization,0.0
,max_features,1.0
,max_bins,255
,categorical_features,'from_dtype'
,monotonic_cst,None


In [96]:
pred = clf.predict(X_test)

In [97]:
confusion_matrix(y_test, pred)

array([[2128, 1548],
       [ 101,   82]])

In [79]:
import numpy as np

In [80]:

print("y_train counts:", np.unique(y_train, return_counts=True))
print("y_test counts :", np.unique(y_test, return_counts=True))

pred = clf.predict(X_test)
print("pred counts   :", np.unique(pred, return_counts=True))

y_train counts: (array([0, 1], dtype=int8), array([11022,   552]))
y_test counts : (array([0, 1], dtype=int8), array([3676,  183]))
pred counts   : (array([0], dtype=int8), array([3859]))


In [81]:
proba1 = clf.predict_proba(X_test)[:, 1]
proba1.min(), proba1.mean(), proba1.max()

(np.float64(0.013079939515372183),
 np.float64(0.04795847548904347),
 np.float64(0.2713574255838325))